# 06 · Bosch 대규모 스케일업 + MCC

**Bosch Production Line** — 실제 생산라인 약 **118만 × 968 변수**, 극불균형(불량 ~0.58%).
SECOM과 같은 VM/FDC 구조를 1000배 규모로 검증. 공식 평가지표 = **MCC**(Matthews Correlation Coefficient).

> 데이터는 Kaggle 인증 필요. 아래 다운로드 후 실행.

## 0. 데이터 준비 (1회)
```bash
pip install kaggle    # ~/.kaggle/kaggle.json 토큰 설정
kaggle competitions download -c bosch-production-line-performance \n    -f train_numeric.csv.zip -p data/raw
cd data/raw && unzip train_numeric.csv.zip
```

In [ ]:
import sys; sys.path.append("..")
from pathlib import Path
p = Path("../data/raw/train_numeric.csv")
assert p.exists(), "train_numeric.csv 없음 — 위 0번 다운로드 먼저"
print("file:", p, f"{p.stat().st_size/1e9:.2f} GB")

## 1. 샘플 로드 (빠른 시작 — 앞 10만 행)

In [ ]:
import numpy as np, pandas as pd
from src.datasets.bosch import load_bosch_sample
X, y = load_bosch_sample(n_rows=100_000)
print("X:", X.shape, "| fail rate:", round(float(y.mean()), 5), f"({int(y.sum())} fails)")
print("missing (mean):", round(float(X.isna().mean().mean()), 3))

## 2. 누수 없는 파이프라인 + 모델 비교 (MCC 포함)
SECOM과 **동일한 코드**를 그대로 재사용 — 스케일만 다름.

In [ ]:
from sklearn.pipeline import Pipeline
from src.features import build_preprocessor
from src.models import get_models
from src.evaluate import cross_val_report
rows = []
for name, clf in get_models(y).items():
    pipe = Pipeline([("pre", build_preprocessor()), ("clf", clf)])
    rows.append(cross_val_report(name, pipe, X, y, n_splits=3))
pd.DataFrame(rows).sort_values("mcc", ascending=False)

## 3. 최적 임계값 MCC (운영 판정 지점)

In [ ]:
from sklearn.model_selection import train_test_split
from src.evaluate import get_scores, best_mcc
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe = Pipeline([("pre", build_preprocessor()), ("clf", get_models(y)["rf"])]).fit(Xtr, ytr)
s = get_scores(pipe, Xte)
mcc, thr = best_mcc(yte, s)
print(f"best MCC = {mcc:.4f}  @ threshold = {thr:.4f}")

## 4. 전체 데이터 — out-of-core
118만 행을 메모리에 못 올리면 청크로 스트리밍(집계/부분학습):


In [ ]:
# from src.datasets.bosch import iter_bosch_chunks
# n, pos = 0, 0
# for Xc, yc in iter_bosch_chunks(chunksize=200_000):
#     n += len(yc); pos += int(yc.sum())
# print("total:", n, "fail rate:", pos/n)

> **포인트**: 극불균형(0.58%)에서 PR-AUC·**MCC**가 정직한 지표. Accuracy는 99.4%라도 무의미.
> 같은 파이프라인이 1.5k(SECOM)→118만(Bosch)로 스케일 = 재현성·신뢰성 증명.